# پروژه تشخیص دستور صوتی
## گام اول آزمایشگاه هوش مصنوعی

**هدف:** دریافت متنِ حاصل از ورودی صوتی و تبدیل آن به یکی از فرمان‌های از پیش تعریف‌شده. این نسخه یک نمونه اولیه است؛ برای تبدیل صدای واقعی به متن، از کتابخانه `SpeechRecognition` و سرویس تشخیص گفتار آنلاین استفاده می‌شود.

### ارتباط با عامل هوشمند
- **حسگر (Sensor):** میکروفون یا ورودی متنِ تشخیص‌داده‌شده از صدا
- **عملگر (Actuator):** اجرای فرمان یا نمایش پاسخ
- **معیار عملکرد:** درصد فرمان‌هایی که درست شناسایی و به عمل مورد انتظار نگاشت می‌شوند
- **محیط (Environment):** کاربر، شرایط صوتی و محیط اجرای برنامه


## ۱. بررسی کتابخانه‌های موردنیاز

در این نمونه، برای آزمایش منطق فرمان‌ها فقط پایتون کافی است. برای دریافت صدا از میکروفون و تبدیل آن به متن، نصب `SpeechRecognition` و `PyAudio` ممکن است لازم باشد. تبدیل آنلاین گفتار به متن به اینترنت نیاز دارد.

در صورت نیاز، در ترمینال VS Code اجرا کنید:
```bash
python -m pip install SpeechRecognition
python -m pip install pyaudio
```
اگر نصب `pyaudio` با خطا مواجه شد، می‌توان فعلاً بخش آزمایش با متن نمونه را اجرا کرد.


In [ ]:
import re
import sys
print("Python version:", sys.version)
print("Python executable:", sys.executable)

try:
    import speech_recognition as sr
    print("SpeechRecognition version:", getattr(sr, "__version__", "unknown"))
except ImportError:
    sr = None
    print("SpeechRecognition is not installed; text-based demo is still available.")


## ۲. تعریف عامل تشخیص دستور

کد زیر مشابه ایده عامل در نمونه کلاسی، ورودی را می‌گیرد، وضعیت/فرمان را بررسی می‌کند و یک عمل مناسب برمی‌گرداند. برای جلوگیری از اجرای ناخواسته، این نسخه فقط فرمان‌های مشخص‌شده را می‌پذیرد.


In [ ]:
class VoiceCommandAgent:
    """عامل ساده برای نگاشت متن تشخیص‌داده‌شده به فرمان."""

    def __init__(self):
        self.commands = {
            "روشن": ("turn_on", "فرمان روشن کردن شناسایی شد."),
            "خاموش": ("turn_off", "فرمان خاموش کردن شناسایی شد."),
            "شروع": ("start", "فرمان شروع شناسایی شد."),
            "توقف": ("stop", "فرمان توقف شناسایی شد."),
            "کمک": ("help", "فرمان کمک شناسایی شد."),
        }

    def normalize(self, text):
        text = text.strip().replace("ي", "ی").replace("ك", "ک")
        text = re.sub(r"\s+", " ", text)
        return text

    def recognize_command(self, text):
        """متن ورودی را به فرمان شناخته‌شده تبدیل می‌کند."""
        normalized = self.normalize(text)
        for keyword, (command, response) in self.commands.items():
            if keyword in normalized:
                return {"input": text, "command": command, "response": response, "success": True}
        return {
            "input": text,
            "command": "unknown",
            "response": "فرمان شناخته نشد؛ یکی از فرمان‌های روشن، خاموش، شروع، توقف یا کمک را بگویید.",
            "success": False,
        }

agent = VoiceCommandAgent()
print("Agent is ready.")


## ۳. آزمایش منطق تشخیص با نمونه‌ها

در این مرحله متن‌هایی را که می‌توانند خروجی یک سیستم تبدیل گفتار به متن باشند آزمایش می‌کنیم. این آزمایش، عملکرد نگاشت فرمان را می‌سنجد و به‌تنهایی معیار دقت تشخیص صوت واقعی نیست.


In [ ]:
test_samples = [
    ("لطفاً دستگاه را روشن کن", "turn_on"),
    ("خاموش کن", "turn_off"),
    ("شروع", "start"),
    ("توقف", "stop"),
    ("کمک می‌خواهم", "help"),
    ("امروز هوا چطور است؟", "unknown"),
]

results = []
for phrase, expected in test_samples:
    result = agent.recognize_command(phrase)
    correct = result["command"] == expected
    results.append(correct)
    print(f"ورودی: {phrase}")
    print(f"فرمان: {result['command']} | نتیجه: {'درست' if correct else 'نادرست'}")
    print("-" * 45)

accuracy = sum(results) / len(results)
print(f"دقت نگاشت متن به فرمان روی نمونه‌های آزمایشی: {accuracy:.1%} ({sum(results)}/{len(results)})")


## ۴. دریافت فرمان از میکروفون (اختیاری)

این سلول به میکروفون و اینترنت نیاز دارد. اگر کتابخانه یا میکروفون در دسترس نیست، سلول را رد کنید و از آزمایش نمونه‌های متنی استفاده کنید.


In [ ]:
def listen_and_recognize(language="fa-IR"):
    if sr is None:
        print("ابتدا SpeechRecognition را نصب کنید.")
        return None

    recognizer = sr.Recognizer()
    try:
        with sr.Microphone() as source:
            print("صحبت کنید؛ فرمان‌هایی مثل روشن، خاموش، شروع، توقف یا کمک را بگویید...")
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)

        # سرویس آنلاین گوگل؛ نیازمند اتصال اینترنت
        transcript = recognizer.recognize_google(audio, language=language)
        print("متن تشخیص‌داده‌شده:", transcript)
        result = agent.recognize_command(transcript)
        print("فرمان نهایی:", result["command"])
        print(result["response"])
        return result
    except Exception as exc:
        print("دریافت یا تشخیص صدا موفق نبود:", type(exc).__name__, str(exc))
        return None

# برای آزمایش با میکروفون، خط زیر را از حالت توضیح خارج کنید:
# listen_and_recognize()


## ۵. محدودیت‌ها و بهبودهای بعدی

- این نمونه، مدل یادگیری ماشین آموزش‌دیده نیست؛ از سرویس تبدیل گفتار به متن و سپس تطبیق کلمات کلیدی استفاده می‌کند.
- تشخیص آنلاین به اینترنت وابسته است و ممکن است برای فارسی در شرایط نویزی خطا داشته باشد.
- آزمایش متنی فقط نگاشت فرمان را می‌سنجد؛ برای گزارش دقت واقعی تشخیص صوت باید مجموعه‌ای از فایل‌های صوتی با برچسب درست تهیه شود.
- گام بعدی می‌تواند افزودن داده صوتی برچسب‌دار، مقایسه فرمان پیش‌بینی‌شده با برچسب واقعی و محاسبه دقت روی داده آزمایشی باشد.
